# Normalize data: HK1-R12Ter
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from itertools import product

from hk1_r12ter_analysis.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR, logger
from hk1_r12ter_analysis.data import sample_normalize_feature_scale_data
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

## Load data and normalize

In [ ]:
sample_key = "Sample"
group_key = "Group"
source_types = ["RBC", "Plasma"]
data_types = ["Proteins", "Metabolites", "Lipids", "Oxylipins"]
value_type = "Intensities"

normalization_inputs = (
    # Sample Normalization, Data Transformation, and Data Scaling
    [None, None, None],
    ["median", None, None],
    ["median", None, "auto"],
)
normalization_strs = [
    "-".join([str(x).lower() for x in norm_inputs]) for norm_inputs in normalization_inputs
]
# For scaling/normalization factor(s), or normalization by a reference sample/feature
reference = None
#  For log transformations
add_constant = None
# DataFrame format axis=0: (Samples, Features) or axis=1: (Features, Samples)
axis = 0

# Directories
input_dirpath = INTERIM_DATA_DIR / "HK1-R12Ter" / "Filtered"
output_dirpath = PROCESSED_DATA_DIR / "HK1-R12Ter" / "Normalized"
output_dirpath.mkdir(parents=True, exist_ok=True)

### Normalize

In [ ]:
normalization_strs = [
    "-".join([str(x).lower() for x in norm_inputs]) for norm_inputs in normalization_inputs
]
index_cols = [key for key in [sample_key, group_key] if key]
for source_type, data_type in product(source_types, data_types):
    # Load data
    logger.debug(f"Processing data for '{source_type}-{data_type}'...")
    filename_args = (source_type, data_type, value_type)
    filename = make_filename(*filename_args)
    df = load_dataframe_from_file(input_dirpath / filename, index_col=index_cols)
    for norm_str in normalization_strs:
        norm_method, transform_method, scaling_method = norm_str.split("-")
        logger.debug(
            f"Processing data using sample normalization: {norm_method}; data transformation: {transform_method}; data scaling: {scaling_method}..."
        )
        df_norm = sample_normalize_feature_scale_data(
            df,
            normalization_method=norm_method,
            transformation_method=transform_method,
            scaling_method=scaling_method,
            axis=axis,
            reference=reference,  # TODO handle multiple provided values
            add_constant=add_constant,  # TODO handle multiple provided values
        )
        output_dirpath_norm = output_dirpath / norm_str
        output_dirpath_norm.mkdir(parents=True, exist_ok=True)
        save_dataframe_to_file(df_norm, output_dirpath_norm / filename, index=True)
        logger.info(
            f"Processed data using sample normalization: {norm_method}; data transformation: {transform_method}; data scaling: {scaling_method}."
        )
    logger.info(f"Processed data for '{source_type}-{data_type}'.")